## Quick Recap

In [1]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
openai_client = OpenAI()

openai_client

In [3]:
system_prompt = "You can make funny and original jokes."
user_prompt = "Tell me a joke about Patrick."

chat_messages = [
    {"role": "developer", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-5-mini',
    input=chat_messages,
)

print(response.output_text)

Why did Patrick bring a ladder to the bar? He heard the drinks were on the house.


### With Function Calling

In [4]:
import random

def make_joke(name):
    jokes = [
        f"Why did {name} bring a pencil to the party? Because he wanted to draw some attention!",
        f"Did you hear about {name}'s bakery? Business is on a roll!",
        f"{name} walked into a library and asked for a burger. The librarian said, 'This is a library.' So {name} whispered, 'Can I get a burger?'",
        f"When {name} does push-ups, the Earth moves down.",
        f"{name} told a chemistry joke... but there was no reaction.",
    ]
    return random.choice(jokes)

print(make_joke("PatrickCmd"))

Did you hear about PatrickCmd's bakery? Business is on a roll!


In [5]:
make_joke_description = {
    "type": "function",
    "name": "make_joke",
    "description": "Generates a random personalized joke using the provided name.",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": "The name to insert into the joke, personalizing the output.",
            }
        },
        "required": ["name"],
        "additionalProperties": False,
    },
}

In [6]:
system_prompt = "You can make funny and original jokes. Find out the user's name to make the joke personalized."

user_prompt = "Tell me a joke about PatrickCmd."

chat_messages = [
    {"role": "developer", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [7]:
response = openai_client.responses.create(
    model='gpt-5-mini',
    input=chat_messages,
    tools=[make_joke_description]
)

In [8]:
response.output

[ResponseReasoningItem(id='rs_0610fcf25f1b9ef400695f53ae7b8c8194b51ca9cd221fee5d', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{"name":"PatrickCmd"}', call_id='call_FLnSK4XSLeK7fwM4qSAwnPfQ', name='make_joke', type='function_call', id='fc_0610fcf25f1b9ef400695f53b170a08194932d20d59aaf1381', status='completed')]

In [9]:
# Save function call outputs for subsequent requests
chat_messages += response.output

In [10]:
chat_messages

[{'role': 'developer',
  'content': "You can make funny and original jokes. Find out the user's name to make the joke personalized."},
 {'role': 'user', 'content': 'Tell me a joke about PatrickCmd.'},
 ResponseReasoningItem(id='rs_0610fcf25f1b9ef400695f53ae7b8c8194b51ca9cd221fee5d', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{"name":"PatrickCmd"}', call_id='call_FLnSK4XSLeK7fwM4qSAwnPfQ', name='make_joke', type='function_call', id='fc_0610fcf25f1b9ef400695f53b170a08194932d20d59aaf1381', status='completed')]

In [11]:
import json

for item in response.output:
    print(f"Item type: {item.type}")
    if item.type == "function_call":
        print(f"Item name: {item.name}")
        print(f"Item arguments: {item.arguments}")
        if item.name == "make_joke":
            # 3. Execute the function logic for make_joke
            joke = make_joke(json.loads(item.arguments))
            print(f"Generated joke: {joke}")
            
            # 4. Provide function call results to the model
            chat_messages.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({
                  "joke": joke
                })
            })

Item type: reasoning
Item type: function_call
Item name: make_joke
Item arguments: {"name":"PatrickCmd"}
Generated joke: {'name': 'PatrickCmd'} walked into a library and asked for a burger. The librarian said, 'This is a library.' So {'name': 'PatrickCmd'} whispered, 'Can I get a burger?'


In [12]:
print("Final input:")
print(chat_messages)


Final input:
[{'role': 'developer', 'content': "You can make funny and original jokes. Find out the user's name to make the joke personalized."}, {'role': 'user', 'content': 'Tell me a joke about PatrickCmd.'}, ResponseReasoningItem(id='rs_0610fcf25f1b9ef400695f53ae7b8c8194b51ca9cd221fee5d', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseFunctionToolCall(arguments='{"name":"PatrickCmd"}', call_id='call_FLnSK4XSLeK7fwM4qSAwnPfQ', name='make_joke', type='function_call', id='fc_0610fcf25f1b9ef400695f53b170a08194932d20d59aaf1381', status='completed'), {'type': 'function_call_output', 'call_id': 'call_FLnSK4XSLeK7fwM4qSAwnPfQ', 'output': '{"joke": "{\'name\': \'PatrickCmd\'} walked into a library and asked for a burger. The librarian said, \'This is a library.\' So {\'name\': \'PatrickCmd\'} whispered, \'Can I get a burger?\'"}'}]


In [13]:
chat_messages

[{'role': 'developer',
  'content': "You can make funny and original jokes. Find out the user's name to make the joke personalized."},
 {'role': 'user', 'content': 'Tell me a joke about PatrickCmd.'},
 ResponseReasoningItem(id='rs_0610fcf25f1b9ef400695f53ae7b8c8194b51ca9cd221fee5d', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{"name":"PatrickCmd"}', call_id='call_FLnSK4XSLeK7fwM4qSAwnPfQ', name='make_joke', type='function_call', id='fc_0610fcf25f1b9ef400695f53b170a08194932d20d59aaf1381', status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_FLnSK4XSLeK7fwM4qSAwnPfQ',
  'output': '{"joke": "{\'name\': \'PatrickCmd\'} walked into a library and asked for a burger. The librarian said, \'This is a library.\' So {\'name\': \'PatrickCmd\'} whispered, \'Can I get a burger?\'"}'}]

In [14]:
response = openai_client.responses.create(
    model="gpt-5-mini",
    instructions="Respond only with a joke generated by a tool.",
    tools=[make_joke_description],
    input=chat_messages,
)

# 5. The model should be able to give a response!
print("Final output:")
print(response.model_dump_json(indent=2))

Final output:
{
  "id": "resp_0610fcf25f1b9ef400695f544f972481948eb0b6f1e70acffc",
  "created_at": 1767855183.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "Respond only with a joke generated by a tool.",
  "metadata": {},
  "model": "gpt-5-mini-2025-08-07",
  "object": "response",
  "output": [
    {
      "id": "msg_0610fcf25f1b9ef400695f545015b881948b9565a8c1614865",
      "content": [
        {
          "annotations": [],
          "text": "PatrickCmd walked into a library and asked for a burger. The librarian said, \"This is a library.\" So PatrickCmd whispered, \"Can I get a burger?\"",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "make_joke",
      "parameters": {
        "type": "object",
        "properties": {
          "na

In [15]:
print("\n" + response.output_text)


PatrickCmd walked into a library and asked for a burger. The librarian said, "This is a library." So PatrickCmd whispered, "Can I get a burger?"


### Orchestrating
For orchestrating, we can use [toyaikit](https://github.com/alexeygrigorev/toyaikit).

ToyAIKit is a minimalistic Python library for building AI assistants powered by Large Language Models (LLMs). It provides a simple yet powerful framework for creating agentic conversational systems with advanced capabilities including function calling, tool integration, and multi-provider support.

> ⚠️ Important: ToyAIKit is great for learning about agents and agentic assistants, but not suitable for production use. For production applications, consider using frameworks like OpenAI Agents SDK, PydanticAI.

In [16]:
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner


In [17]:
tools_obj = Tools()
tools_obj.add_tool(make_joke, make_joke_description)

chat_interface = IPythonChatInterface()
openai_client = OpenAIClient(client=OpenAI())

runner = OpenAIResponsesRunner(
    tools=tools_obj,
    developer_prompt=system_prompt,
    chat_interface=chat_interface,
    llm_client=openai_client
)


In [20]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You can make funny and original jokes. Find out the user's name to make the joke personalized.", role='developer', type=None), EasyInputMessage(content='PatrickCmd', role='user', type=None), ResponseFunctionToolCall(arguments='{"name":"PatrickCmd"}', call_id='call_PaS4GyfCiI9qbEMMSCCCoyeW', name='make_joke', type='function_call', id='fc_0b2073d19df3e77700695f5947648881a09dfe377c118e2766', status='completed'), {'type': 'function_call_output', 'call_id': 'call_PaS4GyfCiI9qbEMMSCCCoyeW', 'output': '"Did you hear about PatrickCmd\'s bakery? Business is on a roll!"'}, ResponseOutputMessage(id='msg_0b2073d19df3e77700695f5948a55c81a0aa9c8adab22893f3', content=[ResponseOutputText(annotations=[], text='Here\'s a joke just for you, PatrickCmd: \n\n"Did you hear about PatrickCmd\'s bakery? Business is on a roll!" \n\nHope that brought a smile!', type='output_text', logprobs=[])], role='assistant', status='completed', type='message'), EasyInputMes

## Django Template Project
### Template
You can download the template that is already working:

```
git clone https://github.com/alexeygrigorev/django_template.git
cd django_template
rm -rf .git
```

In [21]:
!git clone https://github.com/alexeygrigorev/django_template.git

Cloning into 'django_template'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 47 (delta 11), reused 40 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 11.53 KiB | 115.00 KiB/s, done.
Resolving deltas: 100% (11/11), done.


## Coding Agent with Django Template

Now let's create the code for our Agent.

First, we need a function to copy the template into a separate folder

Reference: https://github.com/alexeygrigorev/workshops/tree/main/coding-agent

In [22]:
from openai import OpenAI

from toyaikit.tools import Tools
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner

In [23]:
import os
import shutil

def start(project_name):
    if not project_name:
        print("Project name cannot be empty.")
        return False

    if os.path.exists(project_name):
        print(f"Directory '{project_name}' already exists. Please choose a different name or remove the existing directory.")
        return False

    shutil.copytree('django_template', project_name)
    print(f"Django template copied to '{project_name}' directory.")

    return True

This is how we use it:

```python
project_name = start()
```

Next, we need to define a few functions for the agent: See [tools](./tools.py) module

- Read file
- Write file
- Execute a bash command
- Potentially also: list files and grep


When debugging it, you can use this magic command for automatic file reload:

In [24]:
%load_ext autoreload
%autoreload 2

In [25]:
DEVELOPER_PROMPT = """
You are a coding agent. Your task is to modify the provided Django project template
according to user instructions. You don't tell the user what to do; you do it yourself using the 
available tools. First, think about the sequence of steps you will do, and then 
execute the sequence.
Always ensure changes are consistent with Django best practices and the project’s structure.

## Project Overview

The project is a Django 5.2.4 web application scaffolded with standard best practices. It uses:
- Python 3.12+
- Django 5.2.4 (as specified in pyproject.toml)
- uv for Python environment and dependency management
- SQLite as the default database (see settings.py)
- Standard Django apps and a custom app called myapp
- HTML templates for rendering views
- TailwindCSS for styling

## File Tree


├── .python-version
├── README.md
├── manage.py
├── pyproject.toml
├── uv.lock
├── myapp/
│   ├── __init__.py
│   ├── admin.py
│   ├── apps.py
│   ├── migrations/
│   │   └── __init__.py
│   ├── models.py
│   ├── templates/
│   │   └── home.html
│   ├── tests.py
│   └── views.py
├── myproject/
│   ├── __init__.py
│   ├── asgi.py
│   ├── settings.py
│   ├── urls.py
│   └── wsgi.py
└── templates/
    └── base.html

## Content Description

- manage.py: Standard Django management script for running commands.
- README.md: Setup and run instructions, including use of uv for dependency management.
- pyproject.toml: Project metadata and dependencies (Django 5.2.4).
- uv.lock: Lock file for reproducible Python environments.
- .python-version: Specifies the Python version for the project.
- myapp/: Custom Django app with models, views, admin, tests, and a template (home.html).
  - migrations/: Contains migration files for database schema.
- myproject/: Django project configuration (settings, URLs, WSGI/ASGI entrypoints).
  - settings.py: Configures installed apps, middleware, database (SQLite), templates, etc.
- templates/: Project-level templates, including base.html.

You have full access to modify, add, or remove files and code within this structure using your available tools.


## Additional instructions

- Don't execute "runproject", but you can execute other commands to check if the project is working.
- Make sure you use Tailwind styles for making the result look beautiful
- Use pictograms and emojis when possible. Font-awesome is available
- Avoid putting complex logic to templates - do it on the server side when possible
"""

In [34]:
project_name = input("Enter the new Django project name: ").strip()
start(project_name)

Django template copied to 'pets-store' directory.


True

In [36]:
from pathlib import Path
import tools

project_path = Path(project_name)
agent_tools = tools.AgentTools(project_path)

In [37]:
tools_obj = Tools()
tools_obj.add_tools(agent_tools)

In [38]:
tools_obj.get_tools()

[{'type': 'function',
  'name': 'execute_bash_command',
  'description': 'Execute a bash command in the shell and return its output, error, and exit code. Blocks running the Django development server (runserver).\n\nParameters:\n    command (str): The bash command to execute.\n    cwd (str, optional): Working directory to run the command in, relative to the project directory. Defaults to None.\nReturns:\n    tuple: (stdout (str), stderr (str), returncode (int))',
  'parameters': {'type': 'object',
   'properties': {'command': {'type': 'string',
     'description': 'command parameter'},
    'cwd': {'type': 'string', 'description': 'cwd parameter'}},
   'required': ['command'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'read_file',
  'description': 'Read and return the contents of a file at the given relative filepath.\n\nParameters:\n    filepath (str): Path to the file, relative to the project directory.\nReturns:\n    str: Contents of the file.',
  'parameter

In [39]:
chat_interface = IPythonChatInterface()
openai_client = OpenAIClient(model="gpt-5.2", client=OpenAI())

chat_assistant = OpenAIResponsesRunner(
    tools=tools_obj,
    developer_prompt=DEVELOPER_PROMPT,
    chat_interface=chat_interface,
    llm_client=openai_client
)

In [40]:
chat_assistant.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


...

-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content='\nYou are a coding agent. Your task is to modify the provided Django project template\naccording to user instructions. You don\'t tell the user what to do; you do it yourself using the \navailable tools. First, think about the sequence of steps you will do, and then \nexecute the sequence.\nAlways ensure changes are consistent with Django best practices and the project’s structure.\n\n## Project Overview\n\nThe project is a Django 5.2.4 web application scaffolded with standard best practices. It uses:\n- Python 3.12+\n- Django 5.2.4 (as specified in pyproject.toml)\n- uv for Python environment and dependency management\n- SQLite as the default database (see settings.py)\n- Standard Django apps and a custom app called myapp\n- HTML templates for rendering views\n- TailwindCSS for styling\n\n## File Tree\n\n\n├── .python-version\n├── README.md\n├── manage.py\n├── pyproject.toml\n├── uv.lock\n├── myapp/\n│   ├── __init__.py\n│   ├── admin